# C10｜AI CUP 2026 圍棋：第 1 層重現（Colab，只用 CPU）

從**已存的中間特徵**（CNN 向量、KataGo P1 特徵、各層模型輸出）重算兩場最終交件 CSV，和已上傳的檔比 SHA-256。

| 場次 | 入口 | 預期 SHA-256 |
|---|---|---|
| 競賽二（預測棋力，交件 69） | `repro/comp2_from_features.py` | `c7595f2eb43c185973c55c7b4b03875de41c5ce92327f7042fc3769dbbae36c8` |
| 競賽一（辨識玩家，交件 v2-9） | `repro/comp1_from_features.py` | `36b222d866267abffda03d126d94db9ed5bea3406b2f8b47323622fd5329a026` |

**先把下列 6 個檔放到 Drive 的同一個資料夾**（預設 `MyDrive/C10/report_repro`，可在下一格改）：

- `C10_code.bundle`（程式與 git 內的結果檔，`git bundle`；notebook 會 `git clone` 出工作樹。凍結工具要求輸入檔在 git 裡已 commit 且沒改過，所以不能只解壓 zip）
- `C10_features_layer1_comp1.zip`、`C10_features_layer1_comp2.zip`（不在 git 的特徵檔，不壓縮）
- `C10_features_manifest.json`、`C10_delivery_SHA256SUMS.txt`、`C10_code_bundle_manifest.txt`

執行階段選一般 CPU 即可（不需要 GPU；建議「高 RAM」）。由上到下「全部執行」。

說明：
- 這裡不做任何訓練或特徵抽取，不呼叫 GPU；方法本身都在 repo 既有的程式裡，入口只負責呼叫與比對。
- 凍結工具會比對 Python 3.12.13、numpy 2.5.3、scipy 1.18.1，所以用 `uv` 另建一個 3.12.13 的環境來跑，不用 Colab 內建的 Python。
- 競賽二的工具要求**不要設** `OMP_NUM_THREADS`／`OPENBLAS_NUM_THREADS`／`MKL_NUM_THREADS`；本 notebook 啟動入口時會把這三個變數從子程序的環境拿掉，不另外設任何執行緒變數。競賽一的入口在 Linux 上會在程序內把 numpy 的 OpenBLAS 限成 1 執行緒（原本在 Windows 上既有程式就是這樣跑的）。
- 原本的結果是在 Windows 上算的；Linux 的 numpy／scipy 用不同的 BLAS／libm 建置，浮點數最後一位可能不同，凍結工具的逐位元組接縫會先停下、不產生最終 CSV。所以每支入口**先不加旗標跑一次**（記錄是否逐位元組 PASS）；若停在接縫，**再加 `--tolerant` 跑一次**：只在入口程序內把精確比較換成容許差比較（`repro/_tolerant.py`；數值接縫 |差| ≤ 1e-9 + 1e-9·|參考值|，接縫自己有容許差就用它的；超過仍停），每個接縫的實際最大差異寫進 `tolerant_report.json`。兩次的結果都會印出並寫進 summary。
- **PASS 的判定不變**：只看最終 CSV 的 SHA-256 是否等於已上傳檔。不同時會印出和已上傳檔有幾列不同與前 10 列。
- 交付包不含驗證集答案（`data/local`）；解壓後會檢查樹裡沒有這個資料夾。

In [ ]:
# 0. 參數
DRIVE_DIR = "/content/drive/MyDrive/C10/report_repro"   # Drive 上放交付檔的資料夾
WORK = "/content/c10"          # 解壓後的 repo 樹
OUT = "/content/out"           # 輸出（必須在 repo 樹之外）
VENV = "/content/venv312"      # Python 3.12.13 環境（在 repo 樹之外）
BUNDLE = "C10_code.bundle"     # 程式與 git 內的結果檔（git bundle；先 clone）
ZIPS = ["C10_features_layer1_comp1.zip", "C10_features_layer1_comp2.zip"]   # 不在 git 的特徵檔（後解壓，同一棵樹）

import time
T_START = time.time()
TIMES = {}

In [ ]:
# 1. 掛載 Drive，確認檔案都在
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
need = [BUNDLE] + ZIPS + ["C10_features_manifest.json", "C10_delivery_SHA256SUMS.txt", "C10_code_bundle_manifest.txt"]
# 大檔若是以 7 MB 分塊上傳（Drive 連接器有單檔大小上限），先在本機磁碟依 C10_parts_manifest.json 接回並逐塊、整檔核 SHA-256
import hashlib, json as _json
if not all(os.path.isfile(os.path.join(DRIVE_DIR, n)) for n in [BUNDLE] + ZIPS) and os.path.isfile(os.path.join(DRIVE_DIR, "C10_parts_manifest.json")):
    ASM = "/content/assembled"
    os.makedirs(ASM, exist_ok=True)
    pm = _json.load(open(os.path.join(DRIVE_DIR, "C10_parts_manifest.json"), encoding="utf-8"))
    for name, info in pm.items():
        full = hashlib.sha256()
        with open(os.path.join(ASM, name), "wb") as w:
            for pn, size, h in info["parts"]:
                b = open(os.path.join(DRIVE_DIR, "parts", pn), "rb").read()
                assert len(b) == size and hashlib.sha256(b).hexdigest() == h, f"{pn}: 分塊大小或 SHA-256 不符"
                full.update(b); w.write(b)
        assert full.hexdigest() == info["sha256"], f"{name}: 接回後 SHA-256 不符"
        print(f"接回 {name}：{len(info['parts'])} 塊，SHA-256 {info['sha256'][:12]} 相符")
    for n in ["C10_features_manifest.json", "C10_delivery_SHA256SUMS.txt", "C10_code_bundle_manifest.txt"]:
        shutil.copyfile(os.path.join(DRIVE_DIR, n), os.path.join(ASM, n))
    DRIVE_DIR = ASM
missing = [n for n in need if not os.path.isfile(os.path.join(DRIVE_DIR, n))]
assert not missing, f"Drive 資料夾 {DRIVE_DIR} 少了：{missing}"
for n in need:
    print(f"{os.path.getsize(os.path.join(DRIVE_DIR, n)):>14,}  {n}")

In [ ]:
# 2. 複製到本機磁碟、核 SHA-256；程式用 git clone（bundle）取出工作樹，再把特徵 zip 解壓進同一棵樹
#    為什麼不用 zip：競賽二的凍結工具要求輸入檔「已 commit、沒改過」（要有 git 工作樹和歷史），單純解壓會被拒絕。
import hashlib, shutil, zipfile, subprocess
t0 = time.time()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()

sums = {}
with open(os.path.join(DRIVE_DIR, "C10_delivery_SHA256SUMS.txt"), encoding="utf-8") as f:
    for line in f:
        p = line.split()
        if len(p) >= 2:
            sums[p[1]] = p[0]

assert not os.path.exists(WORK), f"{WORK} 已存在；要重跑請先「中斷連線並刪除執行階段」，或手動刪掉它"
os.makedirs("/content/zips", exist_ok=True)

def fetch(n):
    local = os.path.join("/content/zips", n)
    shutil.copyfile(os.path.join(DRIVE_DIR, n), local)
    got = sha256(local)
    assert got == sums[n], f"{n}: SHA-256 {got} 和 C10_delivery_SHA256SUMS.txt 的 {sums[n]} 不同（上傳不完整？）"
    return local

local = fetch(BUNDLE)
r = subprocess.run(["git", "-c", "core.autocrlf=false", "clone", "-q", local, WORK], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
subprocess.run(["git", "-C", WORK, "config", "core.autocrlf", "false"], check=True)
HEAD = subprocess.run(["git", "-C", WORK, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
want_head = [l.split()[1] for l in open(os.path.join(DRIVE_DIR, "C10_code_bundle_manifest.txt"), encoding="utf-8") if l.startswith("commit")][0]
assert HEAD == want_head, f"clone 出來的 commit {HEAD} 不是 manifest 的 {want_head}"
print(f"{BUNDLE}: sha256 OK，commit {HEAD} -> {WORK}", flush=True)
os.remove(local)
for n in ZIPS:
    local = fetch(n)
    with zipfile.ZipFile(local) as z:
        names = [i.filename for i in z.infolist()]
        bad = [x for x in names if x.startswith(("/", "..")) or "/../" in x or not x.startswith("data/") or x.lower().startswith("data/local")]
        assert not bad, f"{n} 內有不該有的路徑：{bad[:5]}"
        z.extractall(WORK)
    print(f"{n}: sha256 OK，{len(names):,} 個檔 -> {WORK}", flush=True)
    os.remove(local)
assert not os.path.exists(os.path.join(WORK, "data", "local")), "樹裡不應該有 data/local"
st = subprocess.run(["git", "-C", WORK, "status", "--porcelain", "--untracked-files=no"], capture_output=True, text=True).stdout
assert not st.strip(), "git 追蹤的檔被改到了：\n" + st
TIMES["copy + clone + unzip"] = time.time() - t0
print(f"git 工作樹乾淨；{TIMES['copy + clone + unzip']:.0f} 秒")

In [ ]:
# 3. 核 manifest：兩份輸入清單的每個檔（bytes、SHA-256）
import json
t0 = time.time()
with open(os.path.join(DRIVE_DIR, "C10_features_manifest.json"), encoding="utf-8") as f:
    man = json.load(f)
bad = []
for r in man["files"]:
    assert not r["path"].lower().startswith("data/local"), r["path"]
    p = os.path.join(WORK, *r["path"].split("/"))
    if not os.path.isfile(p):
        bad.append((r["path"], "missing"))
    elif os.path.getsize(p) != r["bytes"] or sha256(p) != r["sha256"]:
        bad.append((r["path"], "different"))
print(f"manifest：{len(man['files']):,} 個檔、{man['total_bytes']:,} bytes；不符 {len(bad)} 個")
for x in bad[:20]:
    print("  ", x)
assert not bad, "輸入檔和 manifest 不符，停止"
TIMES["manifest check"] = time.time() - t0
print(f"{TIMES['manifest check']:.0f} 秒")

In [ ]:
# 4. 建 Python 3.12.13 環境（uv），裝 repro/requirements_cpu.txt 的版本
#    requirements_cpu.txt 是 Windows 上 .venv 的實際套件清單。這裡排除：
#      colorama（Windows 上 pytest 的相依；Linux 不需要）
#    清單裡沒有 pywin32 之類只能在 Windows 安裝的套件。
#    torch 是 CPU 版（+cpu），要從 PyTorch 的 CPU 索引抓；第 1 層其實不 import torch。
#    整份裝不起來時（例如某個版本沒有 Linux 輪子），退回只裝第 1 層實際用到的 numpy、scipy、psutil（版本照清單），並印出說明。
import subprocess, re
t0 = time.time()
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-4000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"exit {r.returncode}: {cmd}")
    return r.returncode

sh("pip install -q uv")
sh("uv python install 3.12.13")
sh(f"uv venv {VENV} --python 3.12.13")
PY = f"{VENV}/bin/python"
WINDOWS_ONLY = {"colorama"}
req_src = os.path.join(WORK, "repro", "requirements_cpu.txt")
lines = [l.strip() for l in open(req_src, encoding="utf-8") if l.strip() and not l.startswith("#")]
keep = [l for l in lines if re.split(r"[=<>!~]", l)[0].lower() not in WINDOWS_ONLY]
os.makedirs(OUT, exist_ok=True)
req = os.path.join(OUT, "requirements_linux.txt")
open(req, "w").write("\n".join(keep) + "\n")
INSTALL = "full"
rc = sh(f"uv pip install --python {PY} -r {req} --extra-index-url https://download.pytorch.org/whl/cpu "
        f"--index-strategy unsafe-best-match", check=False)
if rc != 0:
    INSTALL = "minimal (numpy, scipy, psutil)"
    mini = [l for l in keep if re.split(r"[=<>!~]", l)[0].lower() in ("numpy", "scipy", "psutil")]
    print("整份 requirements 裝不起來，改裝第 1 層實際用到的：", mini)
    sh(f"uv pip install --python {PY} " + " ".join(mini))
sh(f"{PY} -c \"import sys, numpy, scipy; print(sys.version.split()[0], numpy.__version__, scipy.__version__)\"")
TIMES["environment"] = time.time() - t0
print("安裝方式：", INSTALL, f"；{TIMES['environment']:.0f} 秒")

In [ ]:
# 5. 跑兩支入口（競賽二先、競賽一後），輸出在 repo 樹之外
#    每支先不加旗標跑一次（逐位元組模式）。若在接縫停下、沒有產生最終 CSV，再加 --tolerant 跑一次（容許模式）。
#    PASS 一律只看最終 CSV 的 SHA-256。
import csv
RESULT = {}
EXPECT = {
    "comp2": ("repro/comp2_from_features.py", "rank/submission_rank.csv", "results/test69/rank/submission_rank.csv",
              "c7595f2eb43c185973c55c7b4b03875de41c5ce92327f7042fc3769dbbae36c8"),
    "comp1": ("repro/comp1_from_features.py", "submission_player.csv", "results/prereg67/apply/submission_player.csv",
              "36b222d866267abffda03d126d94db9ed5bea3406b2f8b47323622fd5329a026"),
}

def child_env():
    env = dict(os.environ)
    for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",   # 入口要求不設（comp2 入口會檢查）
              "PYTHONPATH", "PYTHONHOME"):                                    # 不讓 Colab 內建 Python 的套件混進 3.12.13 環境
        env.pop(v, None)
    env["PYTHONDONTWRITEBYTECODE"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    return env

def rows(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.reader(f))

def diff_rows(got, ref):
    a, b = rows(got), rows(ref)
    n = sum(1 for x, y in zip(a, b) if x != y) + abs(len(a) - len(b))
    return n, len(a), len(b), [(i, x, y) for i, (x, y) in enumerate(zip(a, b)) if x != y][:10]

def run_once(comp, tolerant):
    script, rel_out, rel_ref, want = EXPECT[comp]
    tag = comp + ("_tolerant" if tolerant else "")
    out = os.path.join(OUT, tag)
    shutil.rmtree(out, ignore_errors=True)
    cmd = [PY, "-u", os.path.join(WORK, script), "--root", WORK, "--out", out] + (["--tolerant"] if tolerant else [])
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=WORK, env=child_env(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    log = []
    for line in p.stdout:
        log.append(line)
        print(line, end="", flush=True)
    rc = p.wait()
    sec = time.time() - t0
    open(os.path.join(OUT, f"{tag}_log.txt"), "w", encoding="utf-8").write("".join(log))
    got = os.path.join(out, *rel_out.split("/"))
    ref = os.path.join(WORK, *rel_ref.split("/"))
    res = {"mode": "tolerant" if tolerant else "bytewise", "exit_code": rc, "seconds": round(sec, 1),
           "expected_sha256": want, "produced": os.path.isfile(got)}
    stop = [l.strip() for l in log if l.startswith(("DRIFT", "REFUSED", "FAIL", "WATCHDOG")) or "SystemExit" in l]
    if rc != 0 and not res["produced"]:
        res["stopped_at"] = (stop or [l.strip() for l in log[-3:]])[:3]
    if res["produced"]:
        res["sha256"] = sha256(got)
        res["pass"] = res["sha256"] == want and rc == 0
        if res["sha256"] != want:
            n, la, lb, ex = diff_rows(got, ref)
            res.update(rows_different=n, rows_produced=la, rows_uploaded=lb, first_differences=ex)
    else:
        res["pass"] = False
    if tolerant and os.path.isfile(os.path.join(out, "tolerant_report.json")):
        tr = json.load(open(os.path.join(out, "tolerant_report.json"), encoding="utf-8"))
        res["tolerant_report"] = tr
        shutil.copy(os.path.join(out, "tolerant_report.json"), os.path.join(OUT, f"{comp}_tolerant_report.json"))
    TIMES[tag] = sec
    print("=" * 100)
    print(f"{comp}（{'容許模式 --tolerant' if tolerant else '逐位元組模式'}）: {'PASS' if res['pass'] else 'FAIL'}   exit code {rc}   {sec:.1f} 秒")
    print(f"  sha256   {res.get('sha256', '(沒有產生最終 CSV：在某個檢查或接縫就停了，看上面的訊息)')}")
    print(f"  expected {want}")
    for l in res.get("stopped_at", []):
        print(f"  停在：{l[:300]}")
    if "rows_different" in res:
        print(f"  和已上傳檔不同的列數：{res['rows_different']}（產生 {res['rows_produced']} 列含表頭，已上傳 {res['rows_uploaded']} 列）")
        for i, x, y in res["first_differences"]:
            print(f"    第 {i} 列  產生 {x}  已上傳 {y}")
    if "tolerant_report" in res:
        tr = res["tolerant_report"]
        print(f"  容許模式：檢查 {tr['seams_checked']} 個接縫、放寬 {tr['seams_relaxed']} 個；數值接縫最大絕對差 {tr['numeric_max_abs_diff']:.3g}"
              f"（都在容許值內：{tr['numeric_all_within_tolerance']}）；離散接縫不同列數 {tr['discrete_rows_different']}")
        for e in tr["seams"]:
            what = "相同" if e["strict_equal"] else ("放寬" if e["within_tolerance"] else "超過容許值／有不同列")
            nums = "  ".join(f"{k} {e[k]:.3g}" if isinstance(e[k], float) else f"{k} {e[k]}"
                             for k in ("max_abs_diff", "max_rel_diff", "rows_different") if e.get(k) is not None)
            print(f"    [{what}] {e['seam']}{'（' + e['context'] + '）' if e.get('context') else ''}  {nums}")
    print("=" * 100, flush=True)
    return res

def run(comp):
    strict = run_once(comp, False)
    RESULT[comp] = {"bytewise": strict, "pass_bytewise": strict["pass"], "tolerant": None, "pass_tolerant": None}
    if not strict["produced"]:
        print(f"{comp}: 逐位元組模式沒有產生最終 CSV，改用 --tolerant 再跑一次", flush=True)
        tol = run_once(comp, True)
        RESULT[comp].update(tolerant=tol, pass_tolerant=tol["pass"])
    RESULT[comp]["pass"] = bool(strict["pass"] or RESULT[comp]["pass_tolerant"])

run("comp2")

In [ ]:
run("comp1")

In [ ]:
# 6. 總結：版本、CPU、耗時
import platform
ver = subprocess.run([PY, "-c", "import sys, numpy, scipy, json; print(json.dumps({'python': sys.version.split()[0], 'numpy': numpy.__version__, 'scipy': scipy.__version__}))"],
                     capture_output=True, text=True, env=child_env()).stdout.strip()
cpu = "?"
try:
    cpu = [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
except Exception:
    pass
mem = "?"
try:
    mem = [l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal")][0]
    mem = f"{int(mem) / 1048576:.1f} GB"
except Exception:
    pass
blas = subprocess.run([PY, "-c", "import numpy; numpy.show_config()"], capture_output=True, text=True, env=child_env()).stdout
summary = {"versions": json.loads(ver) if ver else None, "install": INSTALL, "platform": platform.platform(),
           "cpu": cpu, "logical_cpus": os.cpu_count(), "ram": mem,
           "seconds": {k: round(v, 1) for k, v in TIMES.items()}, "seconds_total": round(time.time() - T_START, 1),
           "commit": HEAD, "result": RESULT}
print(json.dumps(summary, ensure_ascii=False, indent=1))
for comp in ("comp2", "comp1"):
    r = RESULT.get(comp, {})
    for mode in ("bytewise", "tolerant"):
        m = r.get(mode)
        if m is None:
            print(f"{comp} {mode}: 沒有跑" + ("（逐位元組模式已產生最終 CSV）" if mode == "tolerant" and r else ""))
            continue
        extra = ""
        if "rows_different" in m:
            extra = f"  和已上傳檔不同 {m['rows_different']} 列"
        if not m["produced"]:
            extra = "  沒有產生最終 CSV：" + "；".join(x[:120] for x in m.get("stopped_at", []))
        if "tolerant_report" in m:
            tr = m["tolerant_report"]
            extra += f"  （放寬 {tr['seams_relaxed']} 個接縫，數值最大差 {tr['numeric_max_abs_diff']:.3g}，離散不同列 {tr['discrete_rows_different']}）"
        print(f"{comp} {mode}: {'PASS' if m['pass'] else 'FAIL'}  {m.get('sha256', '')}  {m['seconds']} 秒{extra}")
    print(f"{comp}: 最終 CSV 和已上傳檔 SHA-256 相同＝{bool(r.get('pass'))}（逐位元組模式 {r.get('pass_bytewise')}、容許模式 {r.get('pass_tolerant')}）")
open(os.path.join(OUT, "colab_layer1_summary.json"), "w", encoding="utf-8").write(json.dumps(summary, ensure_ascii=False, indent=1))
open(os.path.join(OUT, "numpy_config.txt"), "w", encoding="utf-8").write(blas)
# 把紀錄和產生的 CSV 存回 Drive（只有幾百 KB）
dst = os.path.join(DRIVE_DIR, "colab_run_" + time.strftime("%Y%m%d_%H%M%S"))
os.makedirs(dst, exist_ok=True)
for n in ("colab_layer1_summary.json", "numpy_config.txt", "comp1_log.txt", "comp2_log.txt", "comp1_tolerant_log.txt",
          "comp2_tolerant_log.txt", "comp1_tolerant_report.json", "comp2_tolerant_report.json", "requirements_linux.txt"):
    if os.path.isfile(os.path.join(OUT, n)):
        shutil.copy(os.path.join(OUT, n), dst)
for comp, (_, rel_out, _, _) in EXPECT.items():
    for tag in (comp, comp + "_tolerant"):
        p = os.path.join(OUT, tag, *rel_out.split("/"))
        if os.path.isfile(p):
            shutil.copy(p, os.path.join(dst, f"{tag}_" + os.path.basename(p)))
print("紀錄存到", dst)